Import libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#%matplotlib inline
import pandas as pd
import math
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import warnings
from IPython.display import display
import ipywidgets as widgets
pd.options.mode.chained_assignment = None 
from datetime import timezone
import datetime
from suntime import Sun
import pytz
from numpy.linalg import norm
from sklearn.metrics.pairwise import cosine_similarity

Read CSV

In [3]:
dataset = pd.read_csv("10sec-original.csv")

dataset['datetime'] = pd.to_datetime(dataset['datetime'], format='mixed')
lat = 18.47
lon = -66.12
tz = pytz.timezone('America/Puerto_Rico')

In [4]:
dataset

,tagID,datetime,event
0,1621,2017-06-21 08:00:10.850,entering
1,1621,2017-06-21 08:10:33.850,exiting
2,1621,2017-06-21 08:24:48.250,entering
3,1621,2017-06-21 08:51:23.450,entering
4,1621,2017-06-21 08:55:48.300,exiting
...,...,...,...
1855,661,2017-06-22 17:09:08.250,entering
1856,2258,2017-06-22 11:49:10.700,entering
1857,2258,2017-06-22 12:07:42.750,exiting
1858,2258,2017-06-22 16:14:06.350,entering


Data cleaning functions

Separate entry/exit events and convert into flights, takes entry/exit dataset
for a single bee and returns a table.

In [5]:

def classifyLoc(dataset):

    bee = dataset
    
    cur = bee['event']
    nxt = cur.shift(-1)
    cur_dt = bee['datetime']
    nxt_dt = cur_dt.shift(-1)
    #selects all except last
    valid = nxt.notna()
    location = np.select(
        [
            (cur == "entering") & (nxt == "exiting"),
            (cur == "entering") & (nxt == "exiting-entering"),
            
            (cur == "entering-exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering-exiting"),
            
            (cur == "entering") & (nxt == "entering"),
            (cur == "outside") & (nxt == "outside"),
        ],
        ["Inside", "Inside", "Outside", "Outside", "Outside", "Unknown-Inside", "Outside-Unknown"],
        default="Unknown",
    )
            
    t0 = cur_dt.iloc[0]
    age = (cur_dt - t0).dt.days

    #new dataset with inside/outside and time start/end
    BeeTravel = pd.DataFrame(
        {
            "location": location[valid],
            "start": cur_dt[valid],
            "end": nxt_dt[valid],
            "age": age[valid]
        }
    ).reset_index(drop=True)
    
    #classify short outside/insides as part of another group
    BeeTravel['activity'] = "undefined"
    BeeTravel.loc[BeeTravel["location"] == "Inside", "activity"] = "inside"
    duration = (BeeTravel["end"] - BeeTravel["start"]).dt.total_seconds()
    outside = BeeTravel["location"].isin(["Outside"])
   
    #Classify outside flights depending on length
    BeeTravel.loc[outside & (duration < 300), "activity"] = "outside-short"
    BeeTravel.loc[outside & (duration >= 300) & (duration  < 7200), "activity"] = "outside-foraging"
    BeeTravel.loc[outside & (duration >= 7200), "activity"] = "outside-long"
    BeeTravel['tagID'] = dataset['tagID'].iloc[0]

    return BeeTravel



Utilizes the previous function on every individual bee to have a full table.

In [6]:
def cleanData(data):

    activity = []
    
    allActivity = (
        data.sort_values(["tagID","datetime"])
        .groupby("tagID", group_keys = False)[data.columns]
        .apply(classifyLoc)
        .reset_index(drop=True)
    )
    
    
    df = (allActivity[allActivity['location'].isin(["Outside","Outside-Unknown"])]
        [["tagID","start","end"]]
        .rename(columns={'start':'tripStart','end':'tripEnd'})
        .copy()
    )
     
     
    df['duration'] = df['tripEnd'] - df['tripStart']

    return allActivity, df

In [7]:
def fixHourMin(duration):
    hour = int(duration.total_seconds() // 3600)
    mins = int(duration.total_seconds() // 60 % 60)
    if mins < 10:
        mins = f'0{mins}'
    return f'{hour}:{mins}'

Creates a statistical summary of an entire hive based on two tables, all activity and all flights.
Returns a table containing the statistical summary.

Does so by iterating over every beeID separately.

In [8]:
def summaryData(activity,flights):
    
    
    seen = activity.groupby("tagID").agg(
        firstSeen = ('start','min'),
        lastSeen = ('end','max')
    )
    
    seen['firstSeen'] = seen['firstSeen'].dt.floor("s")
    seen['lastSeen'] = seen['lastSeen'].dt.floor("s")
    
    indFlight = flights.groupby("tagID").agg(
        firstFlight = ('tripStart','min'),
        lastFlight = ('tripStart','max'),
        avgTrip = ('duration','mean'),
        medianTrip = ('duration','median'),
        numTrips = ('duration','size')
    )
    
    indFlight['firstFlight'] = indFlight['firstFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['lastFlight'] = indFlight['lastFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['avgTrip'] = indFlight['avgTrip'].map(fixHourMin)
    indFlight['medianTrip'] = indFlight['medianTrip'].map(fixHourMin)
    
    tpd = flights.groupby("tagID").size().rename("tripsPerDay")
    
    full = seen.join(indFlight, how="left").join(tpd, how="left")
    full["tripsPerDay"].fillna(0, inplace = True)
      
    divisor = (full['lastSeen'].dt.date - full['firstSeen'].dt.date).apply(lambda x: x.days) + 1
    full['tripsPerDay'] = full["tripsPerDay"]/divisor
    full['tripsPerDay'] = full["tripsPerDay"].apply(lambda x: x if x > 0 else None)
    
    full['firstSeen'] = full['firstSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    full['lastSeen'] = full['lastSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    longest = flights.groupby('tagID')['duration'].idxmax()
    shortest = flights.groupby('tagID')['duration'].idxmin()
    
    long_df = flights.loc[longest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    short_df = flights.loc[shortest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    
    long_str = (
        long_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + long_df["duration"].map(fixHourMin)
    ).rename("longestFlight")
    
    short_str = (
        short_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + short_df["duration"].map(fixHourMin)
    ).rename("shortestFlight")
    
    full = full.join(long_str, how="left").join(short_str, how="left")
    
    beeSummary = full.rename(columns={"firstSeen":"First Seen", "lastSeen":"Last Seen","firstFlight":"First Flight",
                            "lastFlight":"Last Flight", "longestFlight":"Longest Flight","shortestFlight":"Shortest Flight",
                            "avgTrip":"Average Flight Duration", "numTrips":"# of Flights","tripsPerDay":"Average Flights per Day", "medianTrip":"Median Flight Duration"})
    beeSummary = beeSummary.reset_index()

    return beeSummary

Total summary table creation with values of all hive.

In [9]:
def makeTotalSum(flights):
    #general overview
    totalFlights = len(flights)
    flights['day'] = flights['tripStart'].apply(lambda x: x.date())
    totalPerDay = flights['day'].value_counts().mean()
    totalLength = flights['duration'].mean()
    totalLength = totalLength.total_seconds()/60

    #longest and shortest flights
    longest = flights['duration'].idxmax()
    shortest = flights['duration'].idxmin()

    longestDate = flights.loc[longest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    longestID = flights.loc[longest]['tagID']
    longestLen = flights.loc[longest]['duration'].total_seconds()/60

    shortestDate = flights.loc[shortest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    shortestID = flights.loc[shortest]['tagID']
    shortestLen = flights.loc[shortest]['duration'].total_seconds()/60

    #create tables
    #general info
    totalDict = {"Total Flights":totalFlights, "Avg Flights per Day":totalPerDay, "Avg Length per Day (minutes)":totalLength}
    totalSum = pd.DataFrame.from_dict(totalDict, orient='index')
    totalSum = totalSum.rename(columns={0: "Overall Statistics"})

    #info on longest/shortest trip
    tripDict = {"Classification":["Longest Flight", "Shortest Flight"],"Date":[longestDate, shortestDate], "BeeID":[longestID, shortestID], "Length (minutes)":[longestLen, shortestLen]}
    tripSum = pd.DataFrame.from_dict(tripDict)
    
    return totalSum, tripSum

Obtain sunrise and sunset table for actograph based on previously defined timezone and required days for table.

In [10]:
def getSRSS(dataset, tz, lat, lon):
    dates = dataset['start'].apply(lambda x: x.date()).sort_values()
    dates = dates.unique()
    sunrise = []
    sunset = []
    sun = Sun(lat, lon)
    for d in dates:
        dT = datetime.datetime(d.year, d.month, d.day)
        sunrise.append(sun.get_sunrise_time(dT).astimezone(tz).time())
        sunset.append(sun.get_sunset_time(dT).astimezone(tz).time())

    srss = pd.DataFrame.from_dict({"date":dates,"sunrise":sunrise,"sunset":sunset})
    return srss


Split overnight flights at midnight for plotting reasons

missing = True and timeam/timepm for when a dataset has missing values depending
on dataset requirement.

In [11]:
def separateFlights(flights,missing=False,timeam="08:00:00",timepm="18:00:00"):
    newdata = []
    
    for i in range(len(flights)):
        row = flights.iloc[i].copy()
        row['day'] = row['tripStart'].to_pydatetime().day
        day = row['day']
        month0 = row['tripStart'].to_pydatetime().month
        month1 = row['tripEnd'].to_pydatetime().month
        
        if row['tripStart'].to_pydatetime().day != row['tripEnd'].to_pydatetime().day:
            #first day
            newentry = row.copy()
            newentry['tripEnd'] = row['tripEnd'].replace(hour=23,minute=59,second=59,day=1,month=1)
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)
            #second day
            newday = row['tripEnd'].to_pydatetime().day
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,hour=0,minute=0,second=0,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['day'] = newday
            newentry['date'] = fixDate(row['tripEnd'])
            newdata.append(newentry)
        else:
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)

    if missing:
        cutoffday = str(newentry['tripStart'].date())
        #Extra for missing dataset
        for i in range(len(newdata)):
            cutoffpm = datetime.strptime(cutoffday + " " + timepm,"%Y-%m-%d %H:%M:%S")
            cutoffam = datetime.strptime(cutoffday + " " + timeam,"%Y-%m-%d %H:%M:%S")
            if newdata[i]['tripEnd'] >= cutoffpm:
                newrow = newdata[i].copy()
                newdata[i]['tripEnd'] = cutoffpm
                newdata[i]['Data'] = 'Available'
                newrow['tripStart'] = cutoffpm
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            elif newdata[i]['tripStart'] <= cutoffam:
                newrow = newdata[i].copy()
                newdata[i]['tripStart'] = cutoffam
                newdata[i]['Data'] = 'Available'
                newrow['tripEnd'] = cutoffam
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            else:
                newdata[i]['Data'] = "Available"
    else:
        for i in range(len(newdata)):
            newdata[i]['Data'] = 'Available'
    
    
    flights_mod = pd.concat(newdata, axis = 1)
    flights_mod = flights_mod.transpose().reset_index()
    flights_mod = flights_mod.drop(["index"],axis=1)
    return flights_mod


Apply shapes to time graph based on sunrise/sunset dataset.

Create time plot for bee flights utilizing flights dataset.

In [12]:
#function to change hour column into hour properly

def fixTime(hr):
    if hr < 10:
        return "0" + str(hr) + ":00"
    else:
        return str(hr) + ":00"
        
#function to change date column appropriately

def fixDate(x):
    if x.day > 9 and x.month > 9:
        return f'{x.month}/{x.day}'
    elif x.day > 9 and x.month < 10:
        return f'0{x.month}/{x.day}'
    elif x.day < 10 and x.month > 9:
        return f'{x.month}/0{x.day}'
    else:
        return f'0{x.month}/0{x.day}'

Prepare all new datasets based on csv

In [13]:
allActivity, flights = cleanData(dataset)
beeSummary = summaryData(allActivity,flights)
dt = getSRSS(allActivity, tz, lat, lon)
dt['date'] = dt['date'].apply(lambda x: fixDate(x))
Summary0, Summary1 = makeTotalSum(flights)


flights['date'] = flights['tripStart'].apply(lambda x: fixDate(x))
flights['hour'] = flights['tripStart'].apply(lambda x: x.hour)
flights['day'] = flights['tripStart'].apply(lambda x: x.day)
flights_mod = separateFlights(flights)

In [14]:
flights

,tagID,tripStart,tripEnd,duration,day,date,hour
1,197,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0 days 00:16:39.350000,21,06/21,8
3,197,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0 days 00:21:17.400000,21,06/21,8
7,197,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0 days 01:34:37.900000,21,06/21,11
20,197,2017-06-22 16:13:54.750,2017-06-24 09:07:51.850,1 days 16:53:57.100000,22,06/22,16
22,197,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,0 days 05:20:03.150000,24,06/24,10
...,...,...,...,...,...,...,...
1813,2487,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0 days 00:54:43.200000,22,06/22,8
1818,2487,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,0 days 02:06:43.450000,22,06/22,13
1821,2487,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,0 days 00:29:09.300000,23,06/23,9
1825,2487,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,0 days 01:12:51.950000,24,06/24,8


Divide bees in day-focus/afternoon-focus

In [15]:
def divideBees(dataset):
    hour = dataset['hour'].unique()
    hour.sort()
    split = np.where(hour == 12)[0][0]
    final = {"morning":[],"afternoon":[],"even":[]}
    vectors = {}
    maxcount = dataset[['tagID','hour']].value_counts().max()
    divisor = 10 ** math.ceil(math.log10(maxcount))
    #print(split)
    for i in dataset['tagID'].unique():
        bee = dataset[dataset['tagID'] == i]
        data, bins = np.histogram(bee['hour'], bins=hour,density=True)
        vector = bee['hour'].value_counts().reindex(hour, fill_value=0)
        vector = vector.sort_index()
        vector = vector/divisor
        #print(i)
        #print(data)
        morning = sum(data[0:split])
        afternoon = sum(data[split:])
        if morning > 0.7:
            final['morning'].append(i)
        elif afternoon > 0.7:
            final['afternoon'].append(i)
        else:
            final['even'].append(i)

        vectors[str(i)] = vector.tolist()

    return final,vectors
        

division, vectors = divideBees(flights)

Cosine similarity for collapsed vectors

In [16]:
threshold1 = 0.8
def findSimilar(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold1:
            similars.append(key)
    return(similars)

findSimilar(vectors,1621)

['197', '508', '1237', '1361', '1621', '1780', '2487']

In [17]:
#matrix for every day where 1 = activity, 0 = no activity (matrixes_simp)
#matrix where each value is % of hour spent outside (matrixes)


def createMatrix(df):
    df = df.copy()

    df.sort_values(by='date')
    df['day'] = df['date'].astype('category').cat.codes
    df['tripStart'] = pd.to_datetime(df['tripStart'])
    df['tripEnd']   = pd.to_datetime(df['tripEnd'])
    df['start_hour'] = df['tripStart'].dt.hour
    df['end_hour'] = df['tripEnd'].dt.hour

    n_days = df['day'].nunique()

    result = {}
    result_simple = {}
    result_count = {}

    for tag, g in df.groupby('tagID'):
        matrix = np.zeros((n_days, 24))
        matrix_simple = np.zeros((n_days, 24))
        matrix_count = np.zeros((n_days, 24))

        for index, row in g.iterrows():
            matrix_simple[row.day, row.start_hour:row.end_hour+1] = 1
            matrix_count[row.day, row.start_hour] += 1
            start_h = row.tripStart.hour
            end_h   = row.tripEnd.hour

            for h in range(start_h, end_h + 1):
                hour_start = row.tripStart.floor('D') + pd.Timedelta(hours=h)
                hour_end   = hour_start + pd.Timedelta(hours=1)

                overlap = (
                    min(row.tripEnd, hour_end)
                    - max(row.tripStart, hour_start)
                ).total_seconds()

                if overlap > 0:
                    matrix[row.day, h] += overlap / 3600

                    
        result[str(tag)] = matrix
        result_simple[str(tag)] = matrix_simple
        result_count[str(tag)] = matrix_count

    return result, result_simple, result_count
            
matrixes, matrixes_simp, matrixes_count = createMatrix(flights)

bin size experimentation

In [18]:
flights_count = flights.copy()

In [19]:
flights_count

,tagID,tripStart,tripEnd,duration,day,date,hour
1,197,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0 days 00:16:39.350000,21,06/21,8
3,197,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0 days 00:21:17.400000,21,06/21,8
7,197,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0 days 01:34:37.900000,21,06/21,11
20,197,2017-06-22 16:13:54.750,2017-06-24 09:07:51.850,1 days 16:53:57.100000,22,06/22,16
22,197,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,0 days 05:20:03.150000,24,06/24,10
...,...,...,...,...,...,...,...
1813,2487,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0 days 00:54:43.200000,22,06/22,8
1818,2487,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,0 days 02:06:43.450000,22,06/22,13
1821,2487,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,0 days 00:29:09.300000,23,06/23,9
1825,2487,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,0 days 01:12:51.950000,24,06/24,8


In [20]:
flights_count['half_hour'] = flights_count['tripStart'].apply(lambda x: x.hour + round(x.minute/30)/2)

In [21]:
flights_count['20minute']= flights_count['tripStart'].apply(lambda x: x.hour + math.floor((x.minute/20) * 3)/3)

In [22]:
flights_count['10minute'] = flights_count['tripStart'].apply(lambda x: x.hour + math.floor((x.minute/20) * 6)/6)

In [23]:
flights_count['5minute'] = flights_count['tripStart'].apply(lambda x: x.hour + math.floor((x.minute/20) * 12)/12)

In [24]:
flights_count

,tagID,tripStart,tripEnd,duration,day,date,hour,half_hour,20minute,10minute,5minute
1,197,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0 days 00:16:39.350000,21,06/21,8,8.0,8.333333,8.500000,8.500000
3,197,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0 days 00:21:17.400000,21,06/21,8,8.5,9.666667,9.833333,9.833333
7,197,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0 days 01:34:37.900000,21,06/21,11,11.5,12.666667,12.666667,12.750000
20,197,2017-06-22 16:13:54.750,2017-06-24 09:07:51.850,1 days 16:53:57.100000,22,06/22,16,16.0,16.333333,16.500000,16.583333
22,197,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,0 days 05:20:03.150000,24,06/24,10,10.5,11.000000,11.166667,11.166667
...,...,...,...,...,...,...,...,...,...,...,...
1813,2487,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0 days 00:54:43.200000,22,06/22,8,8.5,9.000000,9.000000,9.083333
1818,2487,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,0 days 02:06:43.450000,22,06/22,13,13.0,13.333333,13.500000,13.500000
1821,2487,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,0 days 00:29:09.300000,23,06/23,9,9.0,9.333333,9.333333,9.333333
1825,2487,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,0 days 01:12:51.950000,24,06/24,8,8.5,9.666667,9.666667,9.750000


In [25]:
flights_count.groupby(['day','hour']).size().describe()

count    76.000000
mean      7.250000
std       5.578829
min       1.000000
25%       3.000000
50%       6.000000
75%      10.000000
max      30.000000
dtype: float64

In [26]:
flights_count.groupby(['day','half_hour']).size().describe()

count    136.000000
mean       4.051471
std        3.151296
min        1.000000
25%        2.000000
50%        3.000000
75%        5.250000
max       19.000000
dtype: float64

In [27]:
flights_count.groupby(['day','20minute']).size().describe()

count    201.000000
mean       2.741294
std        1.820092
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       10.000000
dtype: float64

In [28]:
flights_count.groupby(['day','10minute']).size().describe()

count    303.000000
mean       1.818482
std        1.078103
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        7.000000
dtype: float64

In [29]:
flights_count.groupby(['day','5minute']).size().describe()

count    385.000000
mean       1.431169
std        0.722456
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        5.000000
dtype: float64

In [30]:
#flatten matrix and find cosine similarity of flattened matrix (vector)
threshold2 = 0.6
flattened = {}
for k in matrixes:
    flat = matrixes[k].copy()
    flat = flat.flatten()
    flattened[k] = flat

def findSimilarMatrix(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold2:
            similars.append(key)
    return(similars)
    
findSimilarMatrix(flattened,1621)

['1361', '1621']

In [31]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes_simp,1621,0.7)

['1361', '1621']

In [32]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes,1621,0.7)

['1361', '1621']

In [33]:
def similarFroNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = (current - value)
        n = norm(dif, ord="fro")
        print(key)
        print(n/norm(current,ord="fro"))

print("TIME MATRIX")
similarFroNorm(matrixes,1621)
print("ACTIVITY MATRIX")
similarFroNorm(matrixes_simp,1621)

TIME MATRIX
197
1.14135901106947
508
1.1492148004768075
515
1.248321064240831
516
0.9982112363355458
517
1.047056029599802
638
0.9301910133096223
661
1.0851031801410649
677
0.9683506832625238
930
1.010970740392928
1230
0.9169514980541443
1237
1.22627283149459
1361
1.0884964776682968
1404
1.0069826538839097
1407
1.036153225578213
1415
1.002021507016564
1607
1.057124284736037
1609
1.053216769170626
1621
0.0
1651
0.9999086837955651
1652
1.0540116520124556
1698
0.997143822781802
1722
1.1773292391693118
1780
1.0859047168319231
1786
1.066524506286424
1797
1.1319287257444481
1900
0.9994981680195144
1979
1.0000528028395235
2258
1.1529928906592364
2487
1.0668308421184136
ACTIVITY MATRIX
197
1.0339078862458586
508
1.0170952554312156
515
1.1141720290623112
516
1.0504514628777806
517
1.1596670152276025
638
0.9284766908852594
661
1.0667385033281394
677
0.909717652294684
930
0.909717652294684
1230
0.9284766908852594
1237
1.1596670152276025
1361
0.8709883407113855
1404
0.9649012813540154
1407
1.09858

In [34]:
def similarSpecNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = current - value
        n = norm(dif, ord=2)
        print(key)
        print(n/norm(current, ord=2))

similarSpecNorm(matrixes,1621)

197
1.1375623059542816
508
1.0429719175906194
515
1.145845909957538
516
0.9956182975762087
517
0.9426270116094005
638
0.865573650665705
661
0.981428088808476
677
0.8952873994738805
930
0.9597841349539907
1230
0.9156596695102815
1237
1.1830430571980541
1361
1.1123697010527813
1404
0.9865892223291918
1407
0.9659577068493876
1415
0.9170145872257528
1607
0.9598800615876958
1609
0.9558309913699674
1621
0.0
1651
0.9971143132466597
1652
0.9579312690179217
1698
0.9937932977744129
1722
1.0445971909638023
1780
1.0361951846845432
1786
0.8931897020749682
1797
1.0559209499496267
1900
0.9973748284713884
1979
0.9983537593170445
2258
1.0339659987579037
2487
0.8999157857033094


## FREQUENCY (D1)

In [35]:
flights_freq = flights.groupby(['tagID','day']).size().unstack(fill_value=0)
flights_freq

day,21,22,23,24,25,26,27,28
tagID,,,,,,,,
197,3,1,0,1,0,0,0,0
508,1,0,0,1,2,2,5,1
515,3,3,2,1,0,0,1,0
516,1,1,2,1,1,1,1,1
517,4,8,7,2,2,7,6,0
638,4,4,4,3,0,2,3,0
661,0,1,0,0,0,0,0,0
677,3,0,0,0,0,0,0,0
930,0,1,1,0,1,0,0,0


In [38]:
x = flights_freq.apply(lambda x: [x.mean(), x.std(), stats.variation(x), stats.skew(x)], axis=1)
x

tagID
197     [0.625, 1.0606601717798212, 1.5874507866387544...
508     [1.5, 1.6035674514745464, 1.0, 1.3333333333333...
515     [1.25, 1.2817398889233114, 0.9591663046625438,...
516     [1.125, 0.3535533905932738, 0.2939723678960656...
517     [4.5, 2.9277002188455996, 0.6085806194501846, ...
638     [2.5, 1.6903085094570331, 0.6324555320336759, ...
661     [0.125, 0.3535533905932738, 2.6457513110645907...
677     [0.375, 1.0606601717798212, 2.6457513110645907...
930     [0.375, 0.5175491695067657, 1.2909944487358056...
1230    [7.5, 14.322808982279179, 1.786368134263235, 1...
1237    [3.5, 2.32992949004287, 0.6226998490772392, 0....
1361    [2.375, 1.8468119248354136, 0.7273828926886976...
1404    [0.375, 1.0606601717798212, 2.6457513110645907...
1407    [3.0, 3.927922024247863, 1.2247448713915892, 0...
1415    [0.375, 0.7440238091428449, 1.8559214542766738...
1607    [15.625, 9.226321043622967, 0.5523477165699158...
1609    [3.625, 2.9730936268953445, 0.7671929469409307...
1621    

#### EUCLIDEAN DISTANCE

In [49]:
from scipy.spatial.distance import pdist, squareform

Between two arrays

In [55]:
array = np.array([x.iloc[0], x.iloc[1]])

squareform(pdist(array))

array([[0.        , 1.20928203],
       [1.20928203, 0.        ]])

Between all arrays

In [56]:
array = np.array(x.tolist())
squareform(pdist(array))

array([[ 0.        ,  1.20928203,  1.50097648,  1.7051133 ,  4.78187294,
         3.13424368,  1.53438097,  1.29105575,  1.24873539, 14.93954372,
         3.4339518 ,  2.39269606,  1.29105575,  3.81627217,  0.48465664,
        17.20073592,  3.7876431 ,  3.52449016,  1.36746113,  2.07186811,
         3.23477792,  2.06538049,  4.7520799 ,  0.73188195,  3.42213578,
         1.55552804,  1.53438097,  1.53438097,  1.30345513],
       [ 1.20928203,  0.        ,  1.0368828 ,  1.75352175,  3.67282061,
         2.26545055,  2.65233122,  2.26761417,  1.78804983, 14.08761855,
         2.28869099,  1.30990426,  2.26761417,  2.82316395,  1.67044586,
        16.12926173,  2.65282685,  2.41566593,  2.43881989,  1.26405869,
         2.41815076,  1.22964041,  3.84440883,  0.98521029,  2.28965945,
         1.14941161,  2.65233122,  2.65233122,  0.95583461],
       [ 1.50097648,  1.0368828 ,  0.        ,  2.20918262,  3.71794727,
         1.71110103,  2.92105698,  2.68699087,  1.21577897, 14.53503663,
  

#### WASSERSTEIN

In [52]:
from scipy.stats import wasserstein_distance

Between two arrays

In [59]:
x = flights_freq.apply(lambda x: x.tolist(), axis=1)
x

tagID
197          [3, 1, 0, 1, 0, 0, 0, 0]
508          [1, 0, 0, 1, 2, 2, 5, 1]
515          [3, 3, 2, 1, 0, 0, 1, 0]
516          [1, 1, 2, 1, 1, 1, 1, 1]
517          [4, 8, 7, 2, 2, 7, 6, 0]
638          [4, 4, 4, 3, 0, 2, 3, 0]
661          [0, 1, 0, 0, 0, 0, 0, 0]
677          [3, 0, 0, 0, 0, 0, 0, 0]
930          [0, 1, 1, 0, 1, 0, 0, 0]
1230       [39, 2, 19, 0, 0, 0, 0, 0]
1237         [3, 3, 5, 2, 3, 4, 8, 0]
1361         [5, 1, 3, 1, 2, 2, 5, 0]
1404         [3, 0, 0, 0, 0, 0, 0, 0]
1407         [9, 9, 3, 3, 0, 0, 0, 0]
1415         [1, 2, 0, 0, 0, 0, 0, 0]
1607    [24, 6, 23, 4, 6, 22, 14, 26]
1609         [2, 1, 9, 4, 6, 2, 5, 0]
1621         [6, 3, 6, 2, 7, 2, 0, 0]
1651         [2, 0, 0, 0, 0, 0, 0, 0]
1652         [3, 2, 1, 1, 2, 2, 4, 0]
1698        [11, 2, 2, 4, 0, 0, 0, 0]
1722         [1, 3, 1, 3, 2, 1, 4, 0]
1780        [1, 2, 2, 1, 0, 15, 3, 4]
1786         [4, 3, 0, 0, 0, 0, 0, 0]
1797         [5, 3, 1, 1, 6, 3, 7, 1]
1900         [1, 1, 2, 7, 0, 0, 0, 0]
1979  

Between two arrays

In [60]:
wasserstein_distance(x.iloc[0],x.iloc[1])

np.float64(0.875)

Full matrix

In [69]:
arrays = [wasserstein_distance(i,j) for i in x for j in x]
arr = np.array(arrays).reshape((len(x),len(x)))

In [73]:
arr

array([[ 0.   ,  0.875,  0.625,  0.75 ,  3.875,  1.875,  0.5  ,  0.25 ,
         0.25 ,  6.875,  2.875,  1.75 ,  0.25 ,  2.375,  0.25 , 15.   ,
         3.   ,  2.625,  0.375,  1.25 ,  1.75 ,  1.25 ,  2.875,  0.5  ,
         2.75 ,  0.75 ,  0.5  ,  0.5  ,  0.5  ],
       [ 0.875,  0.   ,  0.5  ,  0.875,  3.   ,  1.25 ,  1.375,  1.125,
         1.125,  6.75 ,  2.   ,  0.875,  1.125,  2.   ,  1.125, 14.125,
         2.125,  1.75 ,  1.25 ,  0.625,  1.375,  0.625,  2.   ,  0.875,
         1.875,  0.625,  1.375,  1.375,  0.625],
       [ 0.625,  0.5  ,  0.   ,  0.875,  3.25 ,  1.25 ,  1.125,  0.875,
         0.875,  6.75 ,  2.25 ,  1.125,  0.875,  2.   ,  0.875, 14.375,
         2.375,  2.   ,  1.   ,  0.625,  1.375,  0.625,  2.25 ,  0.625,
         2.125,  0.875,  1.125,  1.125,  0.125],
       [ 0.75 ,  0.875,  0.875,  0.   ,  3.625,  1.875,  1.   ,  1.   ,
         0.75 ,  7.625,  2.625,  1.5  ,  1.   ,  2.875,  0.75 , 14.5  ,
         2.75 ,  2.625,  0.875,  1.   ,  2.25 ,  1.   ,  2.62

In [72]:
squareform(arr)

array([ 0.875,  0.625,  0.75 ,  3.875,  1.875,  0.5  ,  0.25 ,  0.25 ,
        6.875,  2.875,  1.75 ,  0.25 ,  2.375,  0.25 , 15.   ,  3.   ,
        2.625,  0.375,  1.25 ,  1.75 ,  1.25 ,  2.875,  0.5  ,  2.75 ,
        0.75 ,  0.5  ,  0.5  ,  0.5  ,  0.5  ,  0.875,  3.   ,  1.25 ,
        1.375,  1.125,  1.125,  6.75 ,  2.   ,  0.875,  1.125,  2.   ,
        1.125, 14.125,  2.125,  1.75 ,  1.25 ,  0.625,  1.375,  0.625,
        2.   ,  0.875,  1.875,  0.625,  1.375,  1.375,  0.625,  0.875,
        3.25 ,  1.25 ,  1.125,  0.875,  0.875,  6.75 ,  2.25 ,  1.125,
        0.875,  2.   ,  0.875, 14.375,  2.375,  2.   ,  1.   ,  0.625,
        1.375,  0.625,  2.25 ,  0.625,  2.125,  0.875,  1.125,  1.125,
        0.125,  3.625,  1.875,  1.   ,  1.   ,  0.75 ,  7.625,  2.625,
        1.5  ,  1.   ,  2.875,  0.75 , 14.5  ,  2.75 ,  2.625,  0.875,
        1.   ,  2.25 ,  1.   ,  2.625,  1.25 ,  2.25 ,  1.25 ,  1.   ,
        1.   ,  1.   ,  2.   ,  4.375,  4.125,  4.125,  7.75 ,  1.25 ,
      

In [74]:
np.allclose(arr, arr.T)

True

### DURATION SIMILARITY (D2)

#### TIME BETWEEN FLIGHTS

In [120]:
allActivity['duration'] = (allActivity['end'] - allActivity['start']).apply(lambda x: x.total_seconds())
allActivity['day'] = allActivity['start'].dt.day
allActivity['daydiff'] = (allActivity['end'].dt.hour - allActivity['start'].dt.hour)
dormant = allActivity[(allActivity['location'] == "Inside") & (allActivity['daydiff'] == 0)]

In [121]:
dormant

,location,start,end,age,activity,tagID,duration,day,daydiff
0,Inside,2017-06-21 08:07:56.250,2017-06-21 08:11:24.300,0,inside,197,208.05,21,0
2,Inside,2017-06-21 08:28:03.650,2017-06-21 08:37:01.650,0,inside,197,538.00,21,0
40,Inside,2017-06-26 08:00:17.300,2017-06-26 08:23:00.800,4,inside,508,1363.50,26,0
47,Inside,2017-06-27 09:35:44.700,2017-06-27 09:37:19.850,5,inside,508,95.15,27,0
49,Inside,2017-06-27 10:10:18.650,2017-06-27 10:11:22.450,5,inside,508,63.80,27,0
...,...,...,...,...,...,...,...,...,...
1800,Inside,2017-06-21 09:32:24.700,2017-06-21 09:55:27.800,0,inside,2487,1383.10,21,0
1802,Inside,2017-06-21 10:32:23.300,2017-06-21 10:43:30.450,0,inside,2487,667.15,21,0
1809,Inside,2017-06-22 08:00:57.650,2017-06-22 08:02:33.400,0,inside,2487,95.75,22,0
1812,Inside,2017-06-22 08:03:39.250,2017-06-22 08:23:36.650,0,inside,2487,1197.40,22,0


In [122]:
dormant_times = dormant.groupby(['tagID'])['duration'].apply(list)
dormant_times

tagID
197                                       [208.05, 538.0]
508          [1363.5, 95.15, 63.8, 152.3, 2832.1, 175.45]
515                         [455.35, 261.8, 18.65, 14.35]
517     [344.35, 351.2, 325.0, 515.1, 412.0, 370.0, 70...
638     [750.0, 369.25, 302.4, 376.7, 290.8, 292.35, 2...
661                                                [69.3]
677                               [142.1, 149.45, 1024.9]
1230    [426.4, 48.4, 59.65, 42.5, 66.05, 51.7, 42.2, ...
1237    [824.9, 2178.1, 1051.2, 173.7, 498.05, 165.7, ...
1361    [85.35, 136.8, 273.0, 196.7, 367.35, 544.55, 4...
1404                                              [458.4]
1407    [1281.45, 51.15, 1090.55, 2116.55, 440.3, 113....
1415                                     [672.9, 1953.85]
1607    [752.8, 11.4, 286.45, 22.75, 225.4, 12.5, 650....
1609    [474.1, 307.45, 225.4, 206.5, 156.2, 100.55, 1...
1621    [623.0, 264.85, 873.4, 2718.85, 499.55, 742.15...
1651                                   [1091.35, 1991.75]
1652    

In [123]:
arrays = [wasserstein_distance(i,j) for i in dormant_times for j in dormant_times]
arr = np.array(arrays).reshape((len(dormant_times),len(dormant_times)))
squareform(arr)

array([ 632.50833333,  185.4875    ,  236.37205882, 6408.33928571,
        303.725     ,  258.80833333,  338.4       ,  349.18333333,
        112.92916667,  164.975     ,  407.65625   ,  940.35      ,
        324.36536697,  137.91111111,  415.74558824, 1168.525     ,
        179.85357143,  205.45147059,  493.6375    ,  615.29852941,
         85.325     ,  393.40166667,  343.26666667,  557.555     ,
        621.62916667,  502.49411765, 6382.9452381 ,  712.91666667,
        383.31666667,  659.73981481,  305.00277778,  567.32916667,
        770.95      ,  300.82291667,  825.74166667,  606.99067278,
        542.26944444,  402.08235294, 1041.28333333,  511.4       ,
        566.87254902,  388.10416667,  225.7254902 ,  673.2       ,
        368.16333333,  374.15833333,  632.57      ,  344.13014706,
       6510.27678571,  171.0375    ,  288.72916667,  169.07546296,
        472.5875    ,   90.29166667,  270.8625    ,  347.65625   ,
       1125.8375    ,  160.76135321,  159.48194444,  580.66838

In [125]:
np.allclose(arr, arr.T)

True

#### DURATION OF FLIGHTS

In [126]:
active_times = allActivity[(allActivity['location'] == "Outside") & (allActivity['daydiff'] == 0)].groupby(['tagID'])['duration'].apply(list)
active_times

tagID
197                                      [999.35, 1277.4]
508                                     [2958.65, 1675.5]
515                             [30.9, 88.2, 170.4, 61.4]
516     [71.2, 79.4, 118.0, 55.0, 27.25, 38.85, 60.6, ...
517     [77.0, 1031.85, 1829.0, 49.2, 472.15, 713.2, 7...
638     [1948.55, 1148.3, 853.4, 2086.4, 703.45, 2117....
677                                              [1996.2]
930                                             [86612.8]
1230    [33.8, 44.6, 204.95, 31.5, 27.75, 18.2, 98.15,...
1237    [767.2, 642.3, 170.35, 2405.9, 970.0, 1047.9, ...
1361    [1413.85, 13.0, 571.8, 33.4, 1245.6, 1198.6, 5...
1404                              [146.75, 889.75, 962.6]
1407    [75.85, 639.75, 470.85, 104.1, 97.65, 10.2, 69...
1607    [19.8, 13.75, 26.45, 32.75, 23.3, 20.1, 26.6, ...
1609    [24.1, 738.75, 15.15, 24.35, 115.2, 2876.45, 1...
1621    [854.4, 276.6, 30.6, 1730.8, 13.15, 1671.6, 15...
1651                                      [127.05, 163.5]
1652    

In [127]:
arrays = [wasserstein_distance(i,j) for i in active_times for j in active_times]
arr = np.array(arrays).reshape((len(active_times),len(active_times)))
squareform(arr)

array([1.17870000e+03, 1.05065000e+03, 1.07880278e+03, 6.00937069e+02,
       5.35391667e+02, 8.57825000e+02, 8.54744250e+04, 1.07661272e+03,
       5.36987500e+02, 4.57803571e+02, 4.72008333e+02, 9.30946053e+02,
       1.01262792e+03, 6.99922368e+02, 5.69730556e+02, 9.93100000e+02,
       4.37189286e+02, 1.10498056e+03, 6.51025000e+02, 1.05087000e+03,
       5.52750000e+01, 9.00698333e+02, 1.09017500e+03, 1.01247500e+03,
       5.92000000e+02, 2.22935000e+03, 2.25750278e+03, 1.74159569e+03,
       1.13980833e+03, 6.41575000e+02, 8.42957250e+04, 2.25531272e+03,
       1.14816250e+03, 1.59751786e+03, 1.65070833e+03, 2.10964605e+03,
       2.17214292e+03, 1.66714342e+03, 1.48444167e+03, 2.17180000e+03,
       1.09173929e+03, 2.28368056e+03, 8.79625000e+02, 2.22957000e+03,
       1.16997500e+03, 1.74283167e+03, 2.26887500e+03, 2.19117500e+03,
       1.53415000e+03, 3.00055556e+01, 4.88319828e+02, 1.09182500e+03,
       1.90847500e+03, 8.65250750e+04, 3.95416667e+01, 1.09531250e+03,
      

In [128]:
np.allclose(arr, arr.T)

True

### SCHEDULE SIMILARITY (D3)

Using 20 minute bins

In [129]:
flights_count = flights_count.sort_values(['day','20minute'])
count = flights_count.groupby(['tagID','day','20minute']).size()
idx = pd.MultiIndex.from_product([flights_count.tagID.unique(), flights_count.day.unique(), flights_count['20minute'].unique()])
count = count.reindex(idx).fillna(0)
count
#unstacked = count.unstack(['day','20minute'], fill_value=0)
#unstacked_dict = unstacked.groupby(level=0).apply(lambda df: df.xs(df.name).to_dict()).to_dict()

516  21  8.000000     1.0
         8.333333     0.0
         8.666667     0.0
         9.000000     0.0
         9.333333     0.0
                     ... 
661  28  18.333333    0.0
         18.666667    0.0
         19.333333    0.0
         19.666667    0.0
         19.000000    0.0
Length: 8352, dtype: float64

In [131]:
result = {
    tag: {
        day: g.values.tolist()
        for day, g in tag_group.groupby(level=1)
    }
    for tag, tag_group in count.groupby(level=0)
}
result

{197: {21: [0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  22: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  23: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  24: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,


In [133]:
def compareScheduleDaytoDay(tag1, tag2, result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    values = []
    for (k,v), (k2,v2) in zip(bee1.items(), bee2.items()):
        cosine = np.dot(v, v2) / (norm(v) * norm(v2))
        values.append(cosine)
    return np.nan_to_num(values)

compareScheduleDaytoDay(1621,1361,result)

C:\Users\kashi\AppData\Local\Temp\ipykernel_8964\371221653.py:6: RuntimeWarning: invalid value encountered in scalar divide
  cosine = np.dot(v, v2) / (norm(v) * norm(v2))


array([0.36514837, 0.        , 0.23570226, 0.        , 0.        ,
       0.        , 0.        , 0.        ])

In [134]:
result_mat = {}
for key,val in result.items():
    mat = []
    for day, lst in val.items():
        mat.append(lst)
    npmat = np.array([np.array(xi) for xi in mat])
    result_mat[key] = npmat

result_mat

{197: array([[0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [135]:
def compareMatrixes(tag1,tag2,result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    return cosine_similarity(bee1,bee2)

np.triu(compareMatrixes(1621,1361,result_mat))

array([[0.36514837, 0.        , 0.23570226, 0.        , 0.        ,
        0.        , 0.18257419, 0.        ],
       [0.        , 0.        , 0.33333333, 0.57735027, 0.        ,
        0.        , 0.25819889, 0.        ],
       [0.        , 0.        , 0.23570226, 0.        , 0.        ,
        0.        , 0.36514837, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.31622777, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.1490712 , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ]])

### TREND SIMILARITY (D4)

#### FLIGHT FREQUENCY

In [127]:
flights_freq

day,21,22,23,24,25,26,27,28
tagID,,,,,,,,
197,3,1,0,1,0,0,0,0
508,1,0,0,1,2,2,5,1
515,3,3,2,1,0,0,1,0
516,1,1,2,1,1,1,1,1
517,4,8,7,2,2,7,6,0
638,4,4,4,3,0,2,3,0
661,0,1,0,0,0,0,0,0
677,3,0,0,0,0,0,0,0
930,0,1,1,0,1,0,0,0


In [142]:
from sklearn.linear_model import LinearRegression 

In [143]:
result

{197: {21: [0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  22: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  23: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  24: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   1.0,
   0.0,
   0.0,
   0.0,
   0.0,


In [174]:
flights_stats = {
    k: [
        np.nan_to_num([
            np.mean(day),
            np.std(day),
            stats.variation(day),
            stats.skew(day)
        ])
        for day in v.values()
    ]
    for k, v in result.items()
}
flights_stats

{197: [array([0.08333333, 0.2763854 , 3.31662479, 3.01511345]),
  array([0.02777778, 0.16433555, 5.91607978, 5.74704893]),
  array([0., 0., 0., 0.]),
  array([0.02777778, 0.16433555, 5.91607978, 5.74704893]),
  array([0., 0., 0., 0.]),
  array([0., 0., 0., 0.]),
  array([0., 0., 0., 0.]),
  array([0., 0., 0., 0.])],
 508: [array([0.02777778, 0.16433555, 5.91607978, 5.74704893]),
  array([0., 0., 0., 0.]),
  array([0., 0., 0., 0.]),
  array([0.02777778, 0.16433555, 5.91607978, 5.74704893]),
  array([0.05555556, 0.22906142, 4.12310563, 3.88057   ]),
  array([0.05555556, 0.22906142, 4.12310563, 3.88057   ]),
  array([0.13888889, 0.41851442, 3.01330383, 3.13616574]),
  array([0.02777778, 0.16433555, 5.91607978, 5.74704893])],
 515: [array([0.08333333, 0.2763854 , 3.31662479, 3.01511345]),
  array([0.08333333, 0.2763854 , 3.31662479, 3.01511345]),
  array([0.05555556, 0.22906142, 4.12310563, 3.88057   ]),
  array([0.02777778, 0.16433555, 5.91607978, 5.74704893]),
  array([0., 0., 0., 0.]),


In [175]:
k = flights_freq.apply(lambda x: x.tolist(), axis=1)
k

tagID
197          [3, 1, 0, 1, 0, 0, 0, 0]
508          [1, 0, 0, 1, 2, 2, 5, 1]
515          [3, 3, 2, 1, 0, 0, 1, 0]
516          [1, 1, 2, 1, 1, 1, 1, 1]
517          [4, 8, 7, 2, 2, 7, 6, 0]
638          [4, 4, 4, 3, 0, 2, 3, 0]
661          [0, 1, 0, 0, 0, 0, 0, 0]
677          [3, 0, 0, 0, 0, 0, 0, 0]
930          [0, 1, 1, 0, 1, 0, 0, 0]
1230       [39, 2, 19, 0, 0, 0, 0, 0]
1237         [3, 3, 5, 2, 3, 4, 8, 0]
1361         [5, 1, 3, 1, 2, 2, 5, 0]
1404         [3, 0, 0, 0, 0, 0, 0, 0]
1407         [9, 9, 3, 3, 0, 0, 0, 0]
1415         [1, 2, 0, 0, 0, 0, 0, 0]
1607    [24, 6, 23, 4, 6, 22, 14, 26]
1609         [2, 1, 9, 4, 6, 2, 5, 0]
1621         [6, 3, 6, 2, 7, 2, 0, 0]
1651         [2, 0, 0, 0, 0, 0, 0, 0]
1652         [3, 2, 1, 1, 2, 2, 4, 0]
1698        [11, 2, 2, 4, 0, 0, 0, 0]
1722         [1, 3, 1, 3, 2, 1, 4, 0]
1780        [1, 2, 2, 1, 0, 15, 3, 4]
1786         [4, 3, 0, 0, 0, 0, 0, 0]
1797         [5, 3, 1, 1, 6, 3, 7, 1]
1900         [1, 1, 2, 7, 0, 0, 0, 0]
1979  

In [195]:
y = [item for item in k]
indexes = [i for i,v in k.items()]
x = [[j for j in flights_stats[i]] for i in indexes]
  

In [197]:
y

[[3, 1, 0, 1, 0, 0, 0, 0],
 [1, 0, 0, 1, 2, 2, 5, 1],
 [3, 3, 2, 1, 0, 0, 1, 0],
 [1, 1, 2, 1, 1, 1, 1, 1],
 [4, 8, 7, 2, 2, 7, 6, 0],
 [4, 4, 4, 3, 0, 2, 3, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [3, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 0, 1, 0, 0, 0],
 [39, 2, 19, 0, 0, 0, 0, 0],
 [3, 3, 5, 2, 3, 4, 8, 0],
 [5, 1, 3, 1, 2, 2, 5, 0],
 [3, 0, 0, 0, 0, 0, 0, 0],
 [9, 9, 3, 3, 0, 0, 0, 0],
 [1, 2, 0, 0, 0, 0, 0, 0],
 [24, 6, 23, 4, 6, 22, 14, 26],
 [2, 1, 9, 4, 6, 2, 5, 0],
 [6, 3, 6, 2, 7, 2, 0, 0],
 [2, 0, 0, 0, 0, 0, 0, 0],
 [3, 2, 1, 1, 2, 2, 4, 0],
 [11, 2, 2, 4, 0, 0, 0, 0],
 [1, 3, 1, 3, 2, 1, 4, 0],
 [1, 2, 2, 1, 0, 15, 3, 4],
 [4, 3, 0, 0, 0, 0, 0, 0],
 [5, 3, 1, 1, 6, 3, 7, 1],
 [1, 1, 2, 7, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [3, 3, 1, 2, 0, 0, 0, 0]]

In [199]:
X_flat = np.array([
    stat
    for bee in x
    for stat in bee
])

y_flat = np.array([
    val
    for bee in y
    for val in bee
])

array([ 3,  1,  0,  1,  0,  0,  0,  0,  1,  0,  0,  1,  2,  2,  5,  1,  3,
        3,  2,  1,  0,  0,  1,  0,  1,  1,  2,  1,  1,  1,  1,  1,  4,  8,
        7,  2,  2,  7,  6,  0,  4,  4,  4,  3,  0,  2,  3,  0,  0,  1,  0,
        0,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  0,
        1,  0,  0,  0, 39,  2, 19,  0,  0,  0,  0,  0,  3,  3,  5,  2,  3,
        4,  8,  0,  5,  1,  3,  1,  2,  2,  5,  0,  3,  0,  0,  0,  0,  0,
        0,  0,  9,  9,  3,  3,  0,  0,  0,  0,  1,  2,  0,  0,  0,  0,  0,
        0, 24,  6, 23,  4,  6, 22, 14, 26,  2,  1,  9,  4,  6,  2,  5,  0,
        6,  3,  6,  2,  7,  2,  0,  0,  2,  0,  0,  0,  0,  0,  0,  0,  3,
        2,  1,  1,  2,  2,  4,  0, 11,  2,  2,  4,  0,  0,  0,  0,  1,  3,
        1,  3,  2,  1,  4,  0,  1,  2,  2,  1,  0, 15,  3,  4,  4,  3,  0,
        0,  0,  0,  0,  0,  5,  3,  1,  1,  6,  3,  7,  1,  1,  1,  2,  7,
        0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,
        0,  0,  0,  3,  3

In [202]:
reg = LinearRegression().fit(X_flat, y_flat)
reg.coef_
reg.intercept_

np.float64(7.993605777301127e-15)